In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyect0408") 

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

url_api = "https://itunes.apple.com/search?term=pop+music&limit=200"
response = requests.get(url_api)
datos_json = response.json()['results']


In [0]:

api_data_mapped = []
for item in datos_json:
    api_data_mapped.append({
        "itunes_track_id": item.get("trackId"),
        "track_name": item.get("trackName"),
        "artist_name": item.get("artistName"),
        "genre": item.get("primaryGenreName"),
        "price_usd": item.get("trackPrice"),
        "release_date": item.get("releaseDate")
    })


In [0]:
api_schema = StructType([
    StructField("itunes_track_id", LongType(), True),
    StructField("track_name", StringType(), True),
    StructField("artist_name", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("price_usd", DoubleType(), True),
    StructField("release_date", StringType(), True)
])

In [0]:
df_api = spark.createDataFrame(api_data_mapped, schema=api_schema)

In [0]:
df_api_selected = df_api.select(
    col("itunes_track_id"), col("track_name"), col("artist_name"), 
    col("genre"), col("price_usd"), col("release_date")
)

In [0]:
itunes_final_df = df_api_selected.withColumn("ingestion_date", current_timestamp())

In [0]:
itunes_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.itunes_trends")